# 🎯 CS2 Bot Training - Всё в одном месте
## Скачивание, парсинг и обучение прямо в Colab

**Что делает:**
1. Скачивает демки топовых команд с HLTV прямо в Colab
2. Парсит все действия игроков
3. Обучает AI модель
4. Сохраняет результаты на Google Drive

**Требования:**
- GPU Runtime (Runtime → Change runtime type → T4 GPU)
- Стабильное интернет-соединение

**Преимущества:**
- ✅ Не нужно ничего скачивать на свой ПК
- ✅ Всё работает в облаке
- ✅ Не занимает место на твоём компьютере

## 🔧 Шаг 1: Проверка GPU

In [ ]:
# Проверка GPU (должен быть Tesla T4 или другой)
!nvidia-smi

## 📦 Шаг 2: Установка библиотек

In [ ]:
# Установка всех необходимых библиотек (~2 минуты)
!pip install -q requests beautifulsoup4 awpy pandas pyarrow tqdm torch

print("✅ Библиотеки установлены!")

## 📥 Шаг 3: Клонирование репозитория

In [ ]:
# Удаляем старые копии если есть
!rm -rf /content/cs2-bot-training

# Переходим в корень
%cd /content

# Клонируем репозиторий
!git clone https://github.com/nem1k9/cs2-bot-training.git

# Переходим в папку scripts (там все .py файлы)
%cd cs2-bot-training/scripts

# Проверяем что файлы есть
print("\n✅ Файлы проекта:")
!ls -la

## 🎮 Шаг 4: Скачивание демок с HLTV

**Настройки:**
- `N_DEMOS` - количество демок (рекомендуется 10-20)
- `PLAYER` - имя игрока для фильтрации (или `None` для всех)

**Время:** ~20-40 минут для 10 демок

In [ ]:
# ========== НАСТРОЙКИ ==========
N_DEMOS = 10  # Количество демок (10-20 рекомендуется)
PLAYER = None  # Имя игрока для фильтрации (например "donk", "ZywOo") или None для всех
# ===============================

print(f"🎯 Скачиваем {N_DEMOS} демок с HLTV...")
print(f"⏱️  Это займёт ~{N_DEMOS * 2}-{N_DEMOS * 4} минут...\n")

# Импортируем функцию скачивания
import sys
sys.path.append('/content/cs2-bot-training/scripts')

from download_and_compress import download_and_compress_demos

# Скачиваем демки (без сжатия, т.к. уже в Colab)
import requests
from bs4 import BeautifulSoup
import os
import time
from tqdm import tqdm

# Топовые команды
TOP_TEAMS = [
    "Vitality", "FaZe", "Spirit", "MOUZ", "Natus Vincere",
    "FURIA", "Falcons", "The MongolZ", "G2", "Astralis",
    "3DMAX", "FUT", "PARIVISION", "Aurora"
]

def get_recent_matches(n_matches=20):
    url = "https://www.hltv.org/results"
    
    # Создаём сессию с куками
    session = requests.Session()
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Cache-Control": "max-age=0",
        "Referer": "https://www.hltv.org/"
    }
    
    print(f"🔍 Ищем матчи на HLTV...")
    
    try:
        # Добавляем задержку
        time.sleep(2)
        
        response = session.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        matches = []
        for result in soup.find_all('div', class_='result-con'):
            if len(matches) >= n_matches * 3:
                break
            
            link_tag = result.find('a', class_='a-reset')
            if not link_tag:
                continue
            
            match_url = "https://www.hltv.org" + link_tag['href']
            teams = result.find_all('div', class_='team')
            
            if len(teams) != 2:
                continue
            
            team1 = teams[0].text.strip()
            team2 = teams[1].text.strip()
            
            if team1 in TOP_TEAMS and team2 in TOP_TEAMS:
                matches.append({'url': match_url, 'team1': team1, 'team2': team2})
        
        print(f"✅ Найдено {len(matches)} матчей топовых команд")
        return matches
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        print(f"\n⚠️  HLTV блокирует запросы из Colab!")
        print(f"\n💡 РЕШЕНИЕ 1: Используй сжатый workflow")
        print(f"   Скачай демки на своём ПК и загрузи архив на Drive")
        print(f"   Ссылка: https://colab.research.google.com/github/nem1k9/cs2-bot-training/blob/main/scripts/CS2_Bot_Training_Compressed.ipynb")
        print(f"\n💡 РЕШЕНИЕ 2: Используй VPN в Colab")
        print(f"   Установи VPN расширение и попробуй снова")
        return []

def get_demo_link(match_url):
    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Referer": "https://www.hltv.org/results"
    }
    try:
        time.sleep(2)  # Задержка
        response = session.get(match_url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        demo_link = soup.find('a', text='GOTV Demo')
        if demo_link and 'href' in demo_link.attrs:
            return demo_link['href']
        return None
    except:
        return None

def download_demo(url, save_path):
    try:
        response = requests.get(url, stream=True, timeout=180)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        
        with open(save_path, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc=os.path.basename(save_path)) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        return True
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        if os.path.exists(save_path):
            os.remove(save_path)
        return False

# Создаём папку для демок
!mkdir -p ./demos

# Получаем список матчей
matches = get_recent_matches(n_matches=N_DEMOS)

if matches:
    downloaded = 0
    
    for i, match in enumerate(matches):
        if downloaded >= N_DEMOS:
            break
        
        print(f"\n[{downloaded + 1}/{N_DEMOS}] {match['team1']} vs {match['team2']}")
        
        demo_url = get_demo_link(match['url'])
        if not demo_url:
            print("  ⚠️  Демка недоступна")
            continue
        
        filename = f"demo_{downloaded + 1}.dem"
        filepath = f"./demos/{filename}"
        
        if os.path.exists(filepath):
            print(f"  ✅ Уже есть: {filename}")
            downloaded += 1
            continue
        
        print(f"  📥 Качаем...")
        if download_demo(demo_url, filepath):
            file_size_mb = os.path.getsize(filepath) / 1e6
            if file_size_mb < 100:
                print(f"  ⚠️  Файл слишком маленький ({file_size_mb:.1f} MB)")
                os.remove(filepath)
                continue
            print(f"  ✅ Готово: {file_size_mb:.1f} MB")
            downloaded += 1
        
        time.sleep(5)  # Увеличенная задержка между демками
    
    print(f"\n✅ Скачано {downloaded} демок!")
else:
    print("❌ Не удалось найти матчи")

## 🔍 Шаг 5: Парсинг демок

**Что отслеживается:**
- Движение и позиционирование
- Прицеливание и стрельба
- Использование гранат
- Тактические решения

**Время:** ~10-30 минут

In [ ]:
from parse_demos import build_dataset

print(f"🔄 Парсим демки...")
if PLAYER:
    print(f"🎯 Фокус на игроке: {PLAYER}")
else:
    print(f"🎯 Парсим всех игроков")
    
print(f"⏱️  Это займёт ~10-30 минут...\n")

# Парсим демки
dataset = build_dataset(
    demo_dir="./demos",
    out_path="./dataset.parquet",
    target_player=PLAYER if PLAYER else None
)

if PLAYER:
    print(f"\n✅ Датасет для {PLAYER} готов: {len(dataset):,} тиков")
else:
    print(f"\n✅ Датасет готов: {len(dataset):,} тиков")
    
print(f"📊 Размер датасета: {dataset.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 🚀 Шаг 6: Обучение модели

**Время обучения:** ~2-4 часа

**ВАЖНО:** Не закрывай вкладку браузера!

In [ ]:
from train import train

print(f"🚀 Начинаем обучение модели...")
print(f"⏱️  Это займёт ~2-4 часа...\n")

train()

print(f"\n✅ Обучение завершено!")
print(f"🎯 Модель обучена на про-игре!")

## 💾 Шаг 7: Сохранение результатов на Google Drive

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Создаём папку для результатов
!mkdir -p /content/drive/MyDrive/cs2_bot_results

# Сохраняем модели и датасет
print("💾 Сохраняем результаты на Google Drive...\n")

!cp cs2bot_final.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ cs2bot_final.pt сохранён" || echo "⚠️  cs2bot_final.pt не найден"
!cp checkpoint.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ checkpoint.pt сохранён" || echo "⚠️  checkpoint.pt не найден"
!cp dataset.parquet /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ dataset.parquet сохранён" || echo "⚠️  dataset.parquet не найден"

print("\n✅ Результаты сохранены на Google Drive в папке cs2_bot_results!")
print("\n📁 Что сохранено:")
!ls -lh /content/drive/MyDrive/cs2_bot_results/

## 📥 Шаг 8: Скачать модель на компьютер (опционально)

In [ ]:
# Скачать модель прямо в браузер
from google.colab import files

print("📥 Скачиваем модель...")
files.download('cs2bot_final.pt')
files.download('checkpoint.pt')

print("✅ Модель скачана!")

## 🎯 Готово!

**Что получилось:**
- ✅ Скачаны демки топовых команд с HLTV
- ✅ Распарсены все действия игроков
- ✅ Обучена модель на про-игре
- ✅ Результаты сохранены на Google Drive

**Модель умеет:**
- Двигаться как про-игроки
- Целиться и стрелять
- Использовать гранаты
- Принимать тактические решения

**Это AI обученный на топовых матчах!** 🔥

---

### 💡 Советы:

**Если хочешь больше демок:**
- Измени `N_DEMOS = 20` в Шаге 4
- Запусти заново

**Если хочешь обучить на конкретном игроке:**
- Установи `PLAYER = "donk"` (или другого игрока)
- Запусти заново

**Если закончилось время Colab:**
- Результаты сохранены на Drive
- Можешь продолжить с чекпоинта